In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TEST_TICKERS,
)

# Feature config: LSTM vs XGB use different feature sets
LAG_RETURNS = 5
RSI_PERIOD = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
SEQ_LEN = 30  # LSTM sequence length
XGB_PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
RIDGE_ALPHA = 1.0


def fetch_cnn_fear_greed_index(limit_days=500):
    """Fetch Fear & Greed index (Alternative.me API, used as CNN-style 0-100 sentiment). Returns DataFrame: timestamp, fear_greed."""
    try:
        import urllib.request
        import json
        url = f"https://api.alternative.me/fng/?limit={limit_days}"
        with urllib.request.urlopen(url, timeout=10) as resp:
            data = json.load(resp)
        rows = []
        for d in data.get("data", []):
            ts = pd.to_datetime(int(d["timestamp"]), unit="s").normalize()
            rows.append({"timestamp": ts, "fear_greed": float(d.get("value", np.nan))})
        if not rows:
            return pd.DataFrame(columns=["timestamp", "fear_greed"])
        df = pd.DataFrame(rows).drop_duplicates("timestamp").sort_values("timestamp")
        return df
    except Exception as e:
        print(f"Fear & Greed fetch failed ({e}), using NaN.")
        return pd.DataFrame(columns=["timestamp", "fear_greed"])

In [2]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    """RSI = 100 - 100/(1 + RS), RS = avg gain / avg loss (Wilder)."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Build features: LSTM = return lags, volume lag, RSI, MACD; XGB = VIX, cyclical month, fear_greed. Target = next 21 returns."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        df["vix_lag_1"] = df["vix"].astype(float).shift(1)
    else:
        df["vix_lag_1"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    # LSTM: returns lag, volume lag, RSI, MACD
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + [
        "volume_lag_1", "rsi", "macd_line", "macd_signal"
    ]
    # XGB: VIX, cyclical month, CNN fear and greed index
    feature_cols_xgb = ["vix_lag_1", "month_sin", "month_cos", "fear_greed"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols_lstm + feature_cols_xgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_lstm, feature_cols_xgb, target_cols

def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    """Build (n_seq, seq_len, n_feat) and (n_seq, horizon) for LSTM."""
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [3]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    HAS_LSTM = True
except Exception:
    HAS_LSTM = False

def train_lstm(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=20):
    if not HAS_LSTM or X_seq is None or len(X_seq) < 10:
        return None
    n_feat = X_seq.shape[2]
    model = Sequential([
        LSTM(32, activation="tanh", input_shape=(X_seq.shape[1], n_feat)),
        Dense(horizon),
    ])
    model.compile(optimizer="adam", loss="mse")
    model.fit(X_seq, y_seq, epochs=epochs, verbose=0)
    return model

In [4]:
def train_stack_and_predict(context_df: pd.DataFrame, horizon: int, return_ridge: bool = False):
    """Level 0: XGB (VIX, month, fear_greed) + LSTM (ret lag, volume lag, RSI, MACD). Level 1: Ridge. Return 21 price forecasts; if return_ridge=True also return ridge_models."""
    try:
        feat_df, feature_cols_lstm, feature_cols_xgb, target_cols = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < MIN_TRAIN_STACK + horizon:
        return []
    X_lstm = feat_df[feature_cols_lstm].values.astype(np.float32)
    X_xgb = feat_df[feature_cols_xgb].values.astype(np.float32)
    y = feat_df[target_cols].values.astype(np.float32)
    n = len(y)
    meta_holdout = min(105, (n - horizon) // 2)
    train_end = n - meta_holdout - horizon
    if train_end < SEQ_LEN + horizon + 10:
        return ([], None) if return_ridge else []
    X_lstm_train, X_xgb_train, y_train = X_lstm[:train_end], X_xgb[:train_end], y[:train_end]
    scaler_lstm = StandardScaler()
    scaler_xgb = StandardScaler()
    X_lstm_s_train = scaler_lstm.fit_transform(X_lstm_train)
    X_xgb_s_train = scaler_xgb.fit_transform(X_xgb_train)
    X_lstm_full_s = scaler_lstm.transform(X_lstm)
    X_xgb_full_s = scaler_xgb.transform(X_xgb)
    # Level 0: XGB on XGB features (VIX, month, fear_greed)
    xgb = XGBRegressor(**XGB_PARAMS)
    xgb_multi = MultiOutputRegressor(xgb)
    xgb_multi.fit(X_xgb_s_train, y_train)
    # Level 0: LSTM on LSTM features (ret lag, volume lag, RSI, MACD)
    X_seq, y_seq = build_sequences(X_lstm_s_train, y_train, SEQ_LEN)
    lstm_model = train_lstm(X_seq, y_seq, horizon) if X_seq is not None else None
    meta_X_xgb, meta_X_lstm, meta_y = [], [], []
    for start in range(train_end, n - horizon):
        row_xgb = X_xgb_full_s[start : start + 1]
        xgb_pred = xgb_multi.predict(row_xgb).ravel()
        if lstm_model is not None and start >= SEQ_LEN:
            seq = X_lstm_full_s[start - SEQ_LEN : start]
            if seq.shape[0] == SEQ_LEN:
                lstm_pred = lstm_model.predict(seq.reshape(1, SEQ_LEN, -1), verbose=0).ravel()
            else:
                lstm_pred = xgb_pred
        else:
            lstm_pred = xgb_pred
        meta_X_xgb.append(xgb_pred)
        meta_X_lstm.append(lstm_pred)
        meta_y.append(y[start])
    if len(meta_y) < 5:
        return ([], None) if return_ridge else []
    meta_X_xgb = np.array(meta_X_xgb)
    meta_X_lstm = np.array(meta_X_lstm)
    meta_y = np.array(meta_y)
    ridge_models = []
    for h in range(horizon):
        r = Ridge(alpha=RIDGE_ALPHA)
        r.fit(np.column_stack([meta_X_xgb[:, h], meta_X_lstm[:, h]]), meta_y[:, h])
        ridge_models.append(r)
    last_idx = n - 1
    last_row_xgb = X_xgb_full_s[last_idx : last_idx + 1]
    xgb_21 = xgb_multi.predict(last_row_xgb).ravel()
    if lstm_model is not None and last_idx >= SEQ_LEN:
        seq = X_lstm_full_s[last_idx - SEQ_LEN : last_idx]
        lstm_21 = lstm_model.predict(seq.reshape(1, SEQ_LEN, -1), verbose=0).ravel()
    else:
        lstm_21 = xgb_21
    final_returns = np.array([ridge_models[h].predict([[xgb_21[h], lstm_21[h]]])[0] for h in range(horizon)])
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.cumprod(np.concatenate([[1.0], 1.0 + final_returns]))[1:]
    if return_ridge:
        return [float(p) for p in prices], ridge_models
    return [float(p) for p in prices]


In [5]:
stacked = load_pool_data(with_vix=True, with_volume=True)
fear_greed_df = fetch_cnn_fear_greed_index(limit_days=1500)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = fear_greed_df["timestamp"]
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL    1255
JNJ     1255
JPM     1255
MSFT    1255
SPY     1255
WMT     1255
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-05,AAPL,121.419998,153766600,24.660000,13.0
1,2021-03-08,AAPL,116.360001,154376600,25.469999,13.0
2,2021-03-09,AAPL,121.089996,129525800,24.030001,13.0
3,2021-03-10,AAPL,119.980003,111943300,22.559999,13.0
4,2021-03-11,AAPL,121.959999,103026500,21.910000,13.0
5,2021-03-12,AAPL,121.029999,88105100,20.690001,13.0
6,2021-03-15,AAPL,123.989998,92403800,20.030001,13.0
7,2021-03-16,AAPL,125.570000,115227900,19.790001,13.0
8,2021-03-17,AAPL,124.760002,111932600,19.230000,13.0
9,2021-03-18,AAPL,120.529999,121229700,21.580000,13.0


In [9]:
stacked.describe()

,timestamp,close,volume,vix,fear_greed
count,7530,7530.000000,7.530000e+03,7530.000000,7530.000000
mean,2023-09-01 10:34:31,242.796776,3.504723e+07,19.133777,40.730677
min,2021-03-05 00:00:00,39.430000,2.316700e+06,11.860000,5.000000
25%,2022-06-01 00:00:00,146.037498,1.048508e+07,15.480000,20.000000
50%,2023-08-31 00:00:00,179.259995,2.119925e+07,17.889999,42.000000
75%,2024-11-29 00:00:00,336.582504,5.250452e+07,21.580000,62.000000
max,2026-03-04 00:00:00,695.489990,4.151446e+08,52.330002,94.000000
std,NaN,153.973148,3.281437e+07,5.191873,23.695757


In [6]:
# Same backtest protocol: 60-day test window, 21-day horizon, step 7 days; evaluate on TEST_TICKERS only
model_name = "xgb_lstm_stack"
all_preds = []
for sym in TEST_TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = train_stack_and_predict(context_df, FORECAST_HORIZON)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(
    columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"]
)
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`

symbol
MSFT    126
SPY     126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-05,483.160004,481.734943,0,MSFT
1,2025-12-08,491.019989,482.582225,0,MSFT
2,2025-12-09,492.019989,483.407698,0,MSFT
3,2025-12-10,478.559998,484.021653,0,MSFT
4,2025-12-11,483.470001,484.655659,0,MSFT


In [7]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_xgb_lstm_stack_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_xgb_lstm_stack_pool.parquet")

            model   symbol        MAE       RMSE    MAPE_%
0  xgb_lstm_stack     MSFT  28.089248  32.511009  6.643257
1  xgb_lstm_stack      SPY   9.909873  11.636870  1.444948
2  xgb_lstm_stack  overall  18.999560  24.788529  4.044103
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_xgb_lstm_stack_pool.parquet


In [10]:
# Ridge meta-learner coefficients (one Ridge per horizon step: final_return = coef_xgb * xgb_pred + coef_lstm * lstm_pred + intercept)
sym = TEST_TICKERS[0]
grp = stacked[stacked["symbol"] == sym].sort_values("timestamp").reset_index(drop=True)
n = len(grp)
if n >= TEST_SIZE + MIN_TRAIN_STACK:
    split_idx = n - TEST_SIZE
    context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
    context_df = grp.iloc[:split_idx][context_cols].copy()
    result = train_stack_and_predict(context_df, FORECAST_HORIZON, return_ridge=True)
    if isinstance(result, tuple):
        _, ridge_models = result
        if ridge_models:
            rows = []
            for h, r in enumerate(ridge_models):
                coef = r.coef_
                rows.append({"step": h + 1, "coef_xgb": coef[0], "coef_lstm": coef[1], "intercept": r.intercept_})
            ridge_df = pd.DataFrame(rows)
            print("Ridge coefficients (Level 1 meta-learner): final_return = coef_xgb * xgb_pred + coef_lstm * lstm_pred + intercept")
            print(ridge_df.to_string(index=False))
    else:
        print("Could not obtain Ridge models.")
else:
    print("Not enough data for Ridge coefficients.")

TypeError: train_stack_and_predict() got an unexpected keyword argument 'return_ridge'